# 02-1c. 다른 공장 전력 자료로 Chronos-2 미세조정

02-1의 Chronos-2는 **우리 공장 자료로 추가 학습하지 않고** 예측했습니다. 이 노트북은 **공개된 다른 공장의 15분 전력 자료로 Chronos-2를 미세조정(LoRA)**하면 1시간 앞 예측이 나아지는지 확인합니다.

**왜 다른 공장 자료인가**
- 우리 자료는 7/1~9/14의 74일뿐이라, 우리 자료만으로 미세조정하면 과적합되기 쉽습니다. 부록 파이프라인에서도 우리 자료로 미세조정한 Chronos-2(small)가 추가 학습 없는 경우를 이기지 못했습니다(부록은 `peft`가 없어 LoRA가 아니라 전체 미세조정으로 실행됨).
- 공장 전력은 교대·휴게·주말 패턴이 비슷하므로, 다른 공장 자료로 '공장 전력의 모양'을 더 배우면 도움이 될 수 있습니다.
- 다른 공장 자료는 2016~2018년 자료라 **우리 시험 기간의 정보가 섞일 일이 없고**, 한 번 학습한 모델로 5주를 모두 예측할 수 있습니다.

**사용한 외부 자료** (출처·사용 사유·사용 방법은 README 6장)

| 자료 | 출처 | 라이선스 | 내용 |
|---|---|---|---|
| 독일 산업체 50곳 부하 | Braeuer, F. (2020), Zenodo, doi:10.5281/zenodo.3899018 | CC BY 4.0 | 15분 평균 전력(kW), 2016년 20곳·2017년 30곳, 각 1년 |
| 철강 공장 전력 소비 | Sathishkumar V E 외, UCI Machine Learning Repository #851 (Steel Industry Energy Consumption), doi:10.24432/C52G8C | CC BY 4.0 | 한국 철강 공장 1곳, 15분 사용량(kWh), 2018년 1년 → ×4로 kW 환산 |

두 자료 모두 Chronos-2 사전학습 자료 목록(Chronos-2 논문 표 6)에 없습니다.

> **실행 환경**: Python 3.12.14, `requirements.txt` 환경(peft 포함). 외부 자료 3개 파일은 `data/external/`에 있습니다(출처는 README 6장). CPU로 학습 약 1~1.5시간, 예측 약 15~25분 걸립니다(CPU에 따라 다름. 실제 걸린 시간은 출력에 있음).

## 0. 준비

In [1]:
CHRONOS_REVISION = "29ec3766d36d6f73f0696f85560a422f50e8498c"   # 02-1와 같은 가중치
import hashlib, shutil, sys, tempfile, time, warnings
warnings.filterwarnings("ignore")
warnings.filterwarnings("error", message="`peft` is required")   # peft가 없으면 경고만 내고 전체 미세조정으로 바뀌므로, 이 경고는 오류로 멈춤
from pathlib import Path

import numpy as np
import pandas as pd
import peft
import torch
from chronos import Chronos2Pipeline
from chronos.chronos2.preprocess import from_list_of_dicts
from peft import PeftModel
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error(); hf_logging.disable_progress_bar()

SEED = 42
torch.manual_seed(SEED)
EXT = Path("../data/external"); PRED_DIR = Path("../outputs/predictions"); TAB_DIR = Path("../outputs/tables")
EXT_FILES = {   # 받은 파일이 출처의 파일과 같은지 해시로 확인
    "LoadProfile_20IPs_2016.csv": "48814e75e3532809b14f1efceb1bcdf4afcd2544216661d64f037dc9e35f6235",
    "LoadProfile_30IPs_2017.csv": "6308c16a759ac40261da1a4a0b304a4401172066f5000ffd4fa28dd21609c49a",
    "Steel_industry_data.csv": "9b1cee6f9cb9cd9df2b95814ca90a9a2ff15b7f5f1fba0fae3c643e82072eacc",
}
for f, h in EXT_FILES.items():
    assert (EXT / f).exists(), f"{f}가 없습니다. README 6장의 출처에서 받아 data/external/에 두세요."
    assert hashlib.sha256((EXT / f).read_bytes()).hexdigest() == h, f"{f}의 내용이 다릅니다."

slots = pd.read_csv("../data/processed/slots_15min.csv", encoding="utf-8-sig", parse_dates=["ts", "date"], index_col="ts")
slots["칸번호"] = (slots["시각"] * 4).astype(int)
slots["요일"] = slots.index.dayofweek
slots["하루종류코드"] = slots["하루종류"].map({"평일가동": 0, "토요일": 1, "휴일": 2})
slots["전날가동"] = slots["전날가동"].astype(int)
TEST_STARTS = pd.to_datetime(["2021-08-09", "2021-08-16", "2021-08-23", "2021-08-30", "2021-09-06"])
TEST_ENDS = list(TEST_STARTS[1:]) + [pd.Timestamp("2021-09-15")]
PLAN = ["하루종류코드", "전날가동", "칸번호", "요일"]      # 02-1와 같은 '가동 계획' 정보
CTX = 2016                                               # 02-1와 같은 입력 길이(최근 3주)
print(f"외부 자료 확인 완료 | Python {sys.version.split()[0]} | torch {torch.__version__} | peft {peft.__version__}")

외부 자료 확인 완료 | Python 3.12.14 | torch 2.14.1+cpu | peft 0.21.2


## 1. 외부 자료 정리

- 두 자료의 시각은 '구간 끝' 표기라(파일이 1월 1일 00:15에 시작해 다음 해 1월 1일 00:00에 끝남) 15분 앞당겨 '구간 시작'으로 맞춥니다. 독일 자료에서 서머타임 전환 때 겹치는 시각은 처음 값만 쓰고, 빠진 시각은 앞뒤 값으로 잇습니다(최대 2시간).
- 우리 자료와 같은 이름의 정보(하루 종류, 전날 가동, 시각, 요일)를 만듭니다. 외부 자료에는 생산계획이 없어서, **하루 평균 전력이 그 공장의 하위 10%와 상위 10% 하루 평균의 중간보다 낮은 날을 '쉬는 날'**로 봅니다. 쉬는 날이 아닌 토요일은 '토요일', 나머지는 '평일 가동'입니다.
- 이 '하루 종류'는 그날 전력으로 추정한 값이라, 생산계획으로 정한 우리 자료의 하루 종류와 정의가 다릅니다. 외부 자료는 공장 전력의 모양을 배우는 데만 쓰고, 예측과 평가는 우리 자료의 생산계획 정보로 합니다.

In [2]:
def plan_like(s):
    """외부 공장의 하루 종류·전날 가동을 전력으로 추정 (우리 자료의 생산계획 정보에 대응)"""
    day = s.index.normalize()
    dmean = s.groupby(day).mean()
    lo, hi = dmean.quantile(0.1), dmean.quantile(0.9)
    op = dmean > lo + 0.5 * (hi - lo)
    kind = pd.Series(np.where(~op, 2, np.where(dmean.index.dayofweek == 5, 1, 0)), index=dmean.index)
    prev = op.shift(1, fill_value=True).astype(int)
    return {"하루종류코드": kind.reindex(day).to_numpy(float), "전날가동": prev.reindex(day).to_numpy(float),
            "칸번호": (s.index.hour * 4 + s.index.minute // 15).to_numpy(float), "요일": s.index.dayofweek.to_numpy(float)}

series = {}
for f in ["LoadProfile_20IPs_2016.csv", "LoadProfile_30IPs_2017.csv"]:
    df = pd.read_csv(EXT / f, sep=";", skiprows=1)
    ts = df.pop("Time stamp").str.replace(r"\s+[ab]$", "", regex=True)      # 서머타임 표시(a/b) 제거
    df.index = pd.to_datetime(ts, format="%d.%m.%Y %H:%M:%S") - pd.Timedelta(minutes=15)
    df = df[~df.index.duplicated()].astype(float)
    df = df.reindex(pd.date_range(df.index[0], df.index[-1], freq="15min")).interpolate(limit=8)
    for c in df.columns:
        series[f"독일 {f[-8:-4]}년 {c}"] = df[c]
steel = pd.read_csv(EXT / "Steel_industry_data.csv", encoding="utf-8-sig")
steel.index = pd.to_datetime(steel["date"], format="%d/%m/%Y %H:%M") - pd.Timedelta(minutes=15)
series["한국 철강 공장"] = steel["Usage_kWh"].astype(float) * 4          # 15분 사용량(kWh) × 4 = 평균 전력(kW)

summary = pd.DataFrame({n: {"기간": f"{s.index[0]:%Y-%m-%d} ~ {s.index[-1]:%Y-%m-%d}", "15분 칸 수": len(s),
                            "평균 (kW)": round(s.mean(), 1), "최대 (kW)": round(s.max(), 1),
                            "쉬는 날 비율": round((plan_like(s)["하루종류코드"] == 2).mean(), 2)} for n, s in series.items()}).T
print(f"계열 {len(series)}개, 15분 칸 합계 {summary['15분 칸 수'].sum():,}개")
summary.iloc[[0, 1, 20, 21, -1]]

계열 51개, 15분 칸 합계 1,788,960개


,기간,15분 칸 수,평균 (kW),최대 (kW),쉬는 날 비율
독일 2016년 LG 1,2016-01-01 ~ 2016-12-31,35136,66.7,391.0,0.51
독일 2016년 LG 2,2016-01-01 ~ 2016-12-31,35136,720.4,1793.5,0.34
독일 2017년 LG 01,2017-01-01 ~ 2017-12-31,35040,4.4,8.4,0.42
독일 2017년 LG 02,2017-01-01 ~ 2017-12-31,35040,4.4,8.4,0.59
한국 철강 공장,2018-01-01 ~ 2018-12-30,35040,109.5,628.7,0.35


## 2. LoRA 미세조정

- **LoRA**는 모델의 원래 가중치는 그대로 두고, 일부 층에 작은 보정 행렬만 더해 학습하는 방법입니다. 전체 약 1억 2천만 개 중 약 120만 개(1%)만 학습하므로 적은 자료로도 덜 망가집니다(아래 출력에서 확인).
- 설정: 1시간 앞(4칸) 예측, 입력 길이 2,016칸(02-1와 같음), 학습률 5e-5, 배치 8, 최대 240단계. 개발사 기본 LoRA 설정(r=8)을 씁니다.
- **학습량**: CPU 시간 때문에 개발사 기본값(1,000단계 × 배치 256)보다 훨씬 적게 학습합니다(최대 240단계 × 배치 8 = 학습 창 1,920개, 기본값의 1% 미만). 이 노트북의 결과는 이 짧은 학습 설정에서의 결과입니다.
- **어느 단계의 모델을 쓸지는 우리 학습 기간(7/19~7/30, 시험 전)으로 고릅니다.** 이 기간 가동일 06~20시의 매 정시(정답 4칸이 모두 평가 대상인 경우)를 검증 묶음으로 삼아, 60단계마다 1시간 앞 손실을 재고 가장 낮은 단계를 씁니다. 시험 기간(8/9~)은 쓰지 않습니다.
- **LoRA로 학습했는지 확인**: `peft`가 없으면 Chronos-2는 경고만 내고 전체 미세조정으로 바꿔 학습합니다. 그래서 0절에서 이 경고가 나오면 실행을 멈추게 했고, 학습 뒤에는 모델이 LoRA 구조(PeftModel)인지, 학습한 파라미터 수, 저장된 결과 파일을 출력합니다. 학습 중 체크포인트는 임시 폴더에 두었다가 확인 뒤 지웁니다.

In [3]:
def task(ctx, fut=None):
    d = {"target": ctx["전력"].to_numpy(float), "past_covariates": {c: ctx[c].to_numpy(float) for c in PLAN}}
    if fut is not None:
        d["future_covariates"] = {c: fut[c].to_numpy(float) for c in PLAN}
    return d

train_inputs = from_list_of_dicts([{"target": s.to_numpy(float), "past_covariates": plan_like(s)} for s in series.values()],
                                  prediction_length=4, known_covariates_names=PLAN)
# 검증: 7/19~7/30 가동일 06~20시 매 정시 (정답 4칸이 모두 평가 대상인 경우만). 직전 3주 + 정답 4칸을 한 묶음으로
val_items = []
for t in pd.date_range("2021-07-19", "2021-07-31", freq="h"):
    fut = slots.loc[t:t + pd.Timedelta(minutes=45)]
    if 6 <= t.hour <= 20 and fut["하루종류"].eq("평일가동").all() and fut["정답사용"].all():
        val_items.append(task(slots[slots.index < t + pd.Timedelta(hours=1)].iloc[-(CTX + 4):]))
val_inputs = from_list_of_dicts(val_items, prediction_length=4, known_covariates_names=PLAN)
print(f"학습 계열 {len(train_inputs)}개, 검증 묶음 {len(val_inputs)}개")

from transformers import TrainerCallback
class KeepLog(TrainerCallback):
    """학습 손실·검증 손실 기록을 모아 둠"""
    def __init__(self): self.rows = []
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs: self.rows.append({"step": state.global_step, **logs})
log_cb = KeepLog()

pipe = Chronos2Pipeline.from_pretrained("amazon/chronos-2", revision=CHRONOS_REVISION, device_map="cpu")
CKPT = Path(tempfile.mkdtemp())   # 학습 중 체크포인트(가장 좋은 단계를 고르는 데만 씀)는 임시 폴더에 두고 확인 뒤 지움
t0 = time.time()
ft = pipe.fit(train_inputs, prediction_length=4, validation_inputs=val_inputs, finetune_mode="lora",
              context_length=CTX, learning_rate=5e-5, num_steps=240, batch_size=8,
              output_dir=CKPT, remove_printer_callback=True, disable_tqdm=True,
              eval_steps=60, save_steps=60, logging_steps=60, seed=SEED, data_seed=SEED, callbacks=[log_cb])
print(f"미세조정 완료: {(time.time() - t0) / 60:.0f}분")

# LoRA로 학습했는지 확인: 학습한 모델이 LoRA 구조(PeftModel)인지, 학습한 파라미터가 전체의 일부인지, 저장된 결과가 어댑터 파일인지
assert isinstance(ft.model, PeftModel), "LoRA가 아니라 전체 미세조정으로 학습되었습니다."
n_train, n_all = ft.model.get_nb_trainable_parameters()
saved = sorted(p.name for p in (CKPT / "finetuned-ckpt").glob("*"))
print(f"LoRA 확인: {type(ft.model).__name__} | 학습한 파라미터 {n_train:,}개 / 전체 {n_all:,}개 ({n_train / n_all:.2%})")
print("저장된 결과 파일:", ", ".join(saved))
shutil.rmtree(CKPT, ignore_errors=True)

학습 계열 51개, 검증 묶음 120개


미세조정 완료: 78분
LoRA 확인: PeftModel | 학습한 파라미터 1,206,912개 / 전체 120,684,576개 (1.00%)
저장된 결과 파일: README.md, adapter_config.json, adapter_model.safetensors


In [4]:
# 학습 경과: 단계별 학습 손실과 검증 손실 (가장 낮은 검증 손실의 단계를 씀)
hist = pd.DataFrame(log_cb.rows)
curve = hist.groupby("step").agg(학습손실=("loss", "first"), 검증손실=("eval_loss", "first")).round(4)
print("선택된 단계(검증 손실 최소):", int(curve["검증손실"].idxmin()), "| 최저 검증 손실:", curve["검증손실"].min())
curve

선택된 단계(검증 손실 최소): 60 | 최저 검증 손실: 0.0689


,학습손실,검증손실
step,,
60,0.0881,0.0689
120,0.0956,0.0707
180,0.0786,0.0702
240,0.0812,0.0699


## 3. 1시간 앞 예측 (02-1 2절과 같은 방식)
매 정시에 직전 3주 기록과 가동 계획을 넣고 다음 4칸을 예측합니다. 미세조정 전 모델로 같은 입력을 예측한 02-1 결과와 비교합니다(비교·검정은 02-2 노트북).

In [5]:
t0 = time.time()
out = []
for st, en in zip(TEST_STARTS, TEST_ENDS):
    dfs, fdfs = [], []
    for t in pd.date_range(st, en - pd.Timedelta(hours=1), freq="h"):
        ctx = slots[slots.index < t].iloc[-CTX:]
        fut = slots.loc[t:t + pd.Timedelta(minutes=45)]
        df = pd.DataFrame({"item_id": f"{t:%m%d%H}", "timestamp": ctx.index, "target": ctx["전력"].values})
        fdf = pd.DataFrame({"item_id": f"{t:%m%d%H}", "timestamp": fut.index})
        for c in PLAN:
            df[c] = ctx[c].values.astype(float); fdf[c] = fut[c].values.astype(float)
        dfs.append(df); fdfs.append(fdf)
    pred = ft.predict_df(pd.concat(dfs), future_df=pd.concat(fdfs), prediction_length=4, quantile_levels=[0.1, 0.5, 0.9], batch_size=128)
    pred["발행시각"] = pd.to_datetime("2021" + pred["item_id"], format="%Y%m%d%H")
    out.append(pred)
    print(f"{st:%m/%d} 주 완료, 누적 {time.time() - t0:.0f}초")
ft_hour = pd.concat(out).rename(columns={"timestamp": "대상시각", "predictions": "예측", "0.1": "하위10%", "0.5": "중앙값", "0.9": "상위10%"})
ft_hour = ft_hour[["발행시각", "대상시각", "예측", "하위10%", "중앙값", "상위10%"]]
ft_hour.to_csv(PRED_DIR / "chronos_ft_hour_ahead.csv", encoding="utf-8-sig", index=False)

# 미리 보기: 미세조정 전(02-1)과 같은 칸에서 비교. 평가 칸은 02-2와 같게 '정답으로 쓰는 칸 중 발행 직전 15분 값이 있는 칸'
base = pd.read_csv(PRED_DIR / "chronos_hour_ahead.csv", encoding="utf-8-sig", parse_dates=["발행시각", "대상시각"])
chk = base.merge(ft_hour, on=["발행시각", "대상시각"], suffixes=("_전", "_후")).join(slots[["전력", "정답사용"]], on="대상시각")
has_prev = slots["전력"].reindex(chk["발행시각"] - pd.Timedelta(minutes=15)).notna().to_numpy()
chk = chk[chk["정답사용"] & has_prev]
high = chk[chk["전력"] > 178]                        # 02-2의 고부하 구간(178kW 초과)
mae = lambda d, c: (d[c] - d["전력"]).abs().mean()
print(f"평가 대상 {len(chk):,}칸 | MAE 미세조정 전 {mae(chk, '예측_전'):.2f} → 후 {mae(chk, '예측_후'):.2f}")
print(f"고부하 구간 {len(high):,}칸 | MAE 미세조정 전 {mae(high, '예측_전'):.2f} → 후 {mae(high, '예측_후'):.2f}")

08/09 주 완료, 누적 254초


08/16 주 완료, 누적 562초


08/23 주 완료, 누적 804초


08/30 주 완료, 누적 1090초


09/06 주 완료, 누적 1459초


평가 대상 3,476칸 | MAE 미세조정 전 6.06 → 후 6.10
고부하 구간 324칸 | MAE 미세조정 전 9.37 → 후 9.86


**해석**
- **LoRA로 학습했음을 확인했습니다.** 학습한 모델은 LoRA 구조(PeftModel)이고, 학습한 파라미터는 전체 120,684,576개 중 1,206,912개(1.00%)이며, 저장된 결과도 어댑터 파일(`adapter_config.json`, `adapter_model.safetensors`)입니다.
- 검증 손실은 60단계에서 가장 낮았고(0.0689), 그 뒤로는 0.0699~0.0707로 거의 변하지 않았습니다. 그래서 60단계 모델(학습 창 480개)을 썼습니다.
- 1시간 앞 MAE는 미세조정 전 6.06에서 6.10으로 **나아지지 않았고**, 고부하 구간(178kW 초과)에서는 9.37에서 9.86으로 **오히려 커졌습니다.** 피크를 놓치지 않는 것이 목적이므로 피크 구간에서는 미세조정 전보다 불리합니다.
- 이 결론은 **CPU 제약으로 짧게 학습한 설정**(위 '학습량')에서의 결과입니다. 이 설정에서는 개선이 없었다는 뜻이며, 더 길게 학습했을 때의 결과는 확인하지 못했습니다.
- 평가 칸은 02-2와 같습니다(정답으로 쓰는 칸 중 발행 직전 15분 값이 있는 3,476칸). 그래서 위 숫자는 02-2 표 16·57의 값과 같습니다. 최종 모델과의 비교·검정은 02-2 노트북 3-6절에 있습니다.